# Libraries


In [1]:
import pandas as pd
import numpy as np
import torch
import re
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments, DataCollatorWithPadding)
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
# mounting google drive
# from google.colab import drive

# drive.mount('/content/drive')

# # checking the gpu is collected
# print("PyTorch version:", torch.__version__)
# print("GPU available:", torch.cuda.is_available())

# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))

# Loading the data


In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/namitharoystat/mhc-tc-datasets/both_val.csv
/kaggle/input/datasets/namitharoystat/mhc-tc-datasets/both_train.csv
/kaggle/input/datasets/namitharoystat/mhc-tc-datasets/both_test.csv


In [4]:
test_file_path = "/kaggle/input/datasets/namitharoystat/mhc-tc-datasets/both_test.csv"
val_file_path = "/kaggle/input/datasets/namitharoystat/mhc-tc-datasets/both_val.csv"
train_file_path = "/kaggle/input/datasets/namitharoystat/mhc-tc-datasets/both_train.csv"

In [5]:
train_df = pd.read_csv(train_file_path)
val_df = pd.read_csv(val_file_path)
test_df = pd.read_csv(test_file_path)

# Exploratory Data Analysis

## Checking data type of the datasets

In [6]:
# checking the data type of the train, val and test data sets
print(type(train_df))
print(type(val_df))
print(type(test_df))

# all data happen as dataframe

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


## Checking dimension of the datasets

In [7]:
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# train dats set have 13,727 data points. both train and validation datasets are having 1,488 data points in each.

Train: (13727, 5)
Validation: (1488, 5)
Test: (1488, 5)


## Checking the structure of the data

In [8]:
train_df.head()

,ID,title,post,class_name,class_id
0,691324c4-5c30-44e0-b9e4-45b4f0715e21,a question about the third conditional.,i was making questions for my students and i r...,none,5
1,d4295391-9ca5-4398-b7c8-687e4a984ef1,the epitome of my life,i've recently requested testing accommodations...,adhd,0
2,58937fa5-3c2c-426b-8255-5a140fbab675,what are your favourites offbeat destinations ...,**cambodia** * koh rong: amazing beaches and a...,none,5
3,7daf364c-3b33-4cbe-be37-a214edf9a73e,synesthesia survey (what colour is each month ...,synesthesia. what is synesthesia? according to...,none,5
4,22518271-4bb4-4caf-b683-7305da519288,"science ama series: i’m phil baran, and i’m he...",i’m phil baran and i teach organic chemistry a...,none,5


In [9]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13727 entries, 0 to 13726
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          13727 non-null  object
 1   title       13727 non-null  object
 2   post        13727 non-null  object
 3   class_name  13727 non-null  object
 4   class_id    13727 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 536.3+ KB


In [10]:
train_df.isna().sum()

ID            0
title         0
post          0
class_name    0
class_id      0
dtype: int64

 No missing values

In [11]:
train_df['class_name'].value_counts()

class_name
adhd          2465
depression    2450
anxiety       2422
bipolar       2407
ptsd          2001
none          1982
Name: count, dtype: int64

## Inferences from EDA

1. This is a six-class classification problem.
2. The six classes are label-encoded in the class_id column.
3. The ID and class_name columns are redundant for model training and can therefore be removed.
4. The title and post columns can be concatenated to create a single text column for the NLP model.
5. There are no missing values in the dataset.
6. All six categories have a similar number of observations; therefore, the dataset is approximately balanced.
7. The class label encoding are as follows

| `class_id` | `class_name` |
| ---------: | ------------ |
|          0 | ADHD         |
|          1 | Anxiety      |
|          2 | Bipolar      |
|          3 | Depression   |
|          4 | PTSD         |
|          5 | None         |


## Word cloud for each category

In [12]:
#-------------------------------------------------------------

# Preprocessing

## Concatenating the title and post columns

In [13]:
# Combine title and post
train_df["text"] = train_df["title"] + " " + train_df["post"]
val_df["text"] = val_df["title"] + " " + val_df["post"]
test_df["text"] = test_df["title"] + " " + test_df["post"]

## Removing the redudant columns

In [14]:
train_df = train_df.drop(columns=["ID", "class_name", "title", "post"])
val_df = val_df.drop(columns=["ID", "class_name", "title", "post"])
test_df = test_df.drop(columns=["ID", "class_name", "title", "post"])

In [15]:
train_df.head()

,class_id,text
0,5,a question about the third conditional. i was ...
1,0,the epitome of my life i've recently requested...
2,5,what are your favourites offbeat destinations ...
3,5,synesthesia survey (what colour is each month ...
4,5,"science ama series: i’m phil baran, and i’m he..."


## Remove HTML tags / markup

In [16]:
print(train_df["text"].iloc[0])

a question about the third conditional. i was making questions for my students and i ran into a little tricky grammar. which is correct (focusing on the latter part of the sentence): &amp;#x200b; * if you had traveled to australia yesterday, what wouldn't you have done while you had been there? * if you had traveled to australia yesterday, what wouldn't you have done while you were there? &amp;#x200b; the second \*feels\* right, but i want to make sure.


In [17]:
def clean_text(text):
    # Removing HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove Reddit-style markdown formatting
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)  # [text](url)

    return text

In [18]:
# applying the clean text function on all 3 data frames
train_df["text"] = train_df["text"].apply(clean_text)
val_df["text"] = val_df["text"].apply(clean_text)
test_df["text"] = test_df["text"].apply(clean_text)

## Replacing Reddit usernames

In [19]:
train_df["text"] = train_df["text"].apply(
    lambda x: re.sub(r'u/\w+', 'USER', x)
)

val_df["text"] = val_df["text"].apply(
    lambda x: re.sub(r'u/\w+', 'USER', x)
)

test_df["text"] = test_df["text"].apply(
    lambda x: re.sub(r'u/\w+', 'USER', x)
)

## Replacing mentions to @user

In [20]:
train_df["text"] = train_df["text"].apply(
    lambda x: re.sub(r'@\w+', '@USER', x)
)

val_df["text"] = val_df["text"].apply(
    lambda x: re.sub(r'@\w+', '@USER', x)
)

test_df["text"] = test_df["text"].apply(
    lambda x: re.sub(r'@\w+', '@USER', x)
)

## Dealing the special characters

In [21]:
special_chars = Counter(
    char
    for text in train_df["text"]
    for char in text
    if not char.isalnum() and not char.isspace()
)

special_chars.most_common(30)

[('.', 172147),
 (',', 107361),
 ("'", 71424),
 ('’', 22768),
 ('*', 20182),
 ('-', 18703),
 ('?', 15354),
 ('"', 15245),
 (')', 12020),
 ('(', 11397),
 (':', 9287),
 ('!', 8973),
 ('/', 8007),
 (';', 6058),
 ('[', 5602),
 ('|', 4980),
 ('&', 2950),
 ('“', 2162),
 ('”', 2124),
 ('_', 1909),
 ('#', 1559),
 ('%', 978),
 ('$', 942),
 ('\\', 707),
 (']', 678),
 ('+', 650),
 ('=', 560),
 ('—', 451),
 ('~', 426),
 ('‘', 378)]

The most common punctuation marks are ., ,, ", ', (, ), :, !, and ?. These characters may provide useful contextual and semantic information to a Transformer-based language model. Although some special characters occur less frequently, with frequencies of only a few hundred occurrences, low frequency alone does not provide sufficient evidence that these characters are noise or problematic. Such symbols may have legitimate uses within Reddit posts and could contribute to the meaning or context of particular text segments. Hence, no action was taken to remove special characters.

## Removing extra whitespace

In [22]:
train_df["text"] = train_df["text"].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

val_df["text"] = val_df["text"].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

test_df["text"] = test_df["text"].apply(
    lambda x: re.sub(r'\s+', ' ', x).strip()
)

# Tokenisation

In [23]:
# checking the data before tokenisation
train_df.head()
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13727 entries, 0 to 13726
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   class_id  13727 non-null  int64 
 1   text      13727 non-null  object
dtypes: int64(1), object(1)
memory usage: 214.6+ KB


In [24]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [25]:
# texting on one line
text = train_df["text"].iloc[0]

tokens = tokenizer.tokenize(text)

print(tokens)
#Ġ is used to indicate that the token starts after a whitespace/space.

['a', 'Ġquestion', 'Ġabout', 'Ġthe', 'Ġthird', 'Ġconditional', '.', 'Ġi', 'Ġwas', 'Ġmaking', 'Ġquestions', 'Ġfor', 'Ġmy', 'Ġstudents', 'Ġand', 'Ġi', 'Ġran', 'Ġinto', 'Ġa', 'Ġlittle', 'Ġtricky', 'Ġgrammar', '.', 'Ġwhich', 'Ġis', 'Ġcorrect', 'Ġ(', 'focus', 'ing', 'Ġon', 'Ġthe', 'Ġlatter', 'Ġpart', 'Ġof', 'Ġthe', 'Ġsentence', '):', 'Ġ&', 'amp', ';', '#', 'x', '200', 'b', ';', 'Ġ*', 'Ġif', 'Ġyou', 'Ġhad', 'Ġtraveled', 'Ġto', 'Ġaust', 'ral', 'ia', 'Ġyesterday', ',', 'Ġwhat', 'Ġwouldn', "'t", 'Ġyou', 'Ġhave', 'Ġdone', 'Ġwhile', 'Ġyou', 'Ġhad', 'Ġbeen', 'Ġthere', '?', 'Ġ*', 'Ġif', 'Ġyou', 'Ġhad', 'Ġtraveled', 'Ġto', 'Ġaust', 'ral', 'ia', 'Ġyesterday', ',', 'Ġwhat', 'Ġwouldn', "'t", 'Ġyou', 'Ġhave', 'Ġdone', 'Ġwhile', 'Ġyou', 'Ġwere', 'Ġthere', '?', 'Ġ&', 'amp', ';', '#', 'x', '200', 'b', ';', 'Ġthe', 'Ġsecond', 'Ġ\\*', 'fe', 'els', '\\*', 'Ġright', ',', 'Ġbut', 'Ġi', 'Ġwant', 'Ġto', 'Ġmake', 'Ġsure', '.']


In [26]:
train_tokens = tokenizer(
    train_df["text"].tolist(),
    truncation=True
)

val_tokens = tokenizer(
    val_df["text"].tolist(),
    truncation=True
)

test_tokens = tokenizer(
    test_df["text"].tolist(),
    truncation=True
)

In [27]:
# print(train_tokens.keys())
# print(len(train_tokens["input_ids"]))
# print(train_tokens["input_ids"][0])

In [28]:
# checking the pad_token and token id modernbert tokeniser uses.
print(tokenizer.pad_token)
print(tokenizer.pad_token_id)

[PAD]
50283


Tokenized the cleaned text data using the ModernBERT tokenizer, which converted the text into numerical token IDs.padding is not apllied at this stage; will use dynamic padding during batch preparation.

# Adding labels

In [29]:
train_tokens["labels"] = train_df["class_id"].tolist()
val_tokens["labels"] = val_df["class_id"].tolist()
test_tokens["labels"] = test_df["class_id"].tolist()

In [30]:
#print(train_tokens.keys())

# Creating the hugging face data set

Initially, the data was loaded as pandas DataFrames containing the text and class labels. After preprocessing and tokenization, the text was converted into token IDs. The tokenized data was then converted into Hugging Face Dataset objects so that it could be used with the Hugging Face Transformers training pipeline and related tools.


In [31]:
from datasets import Dataset

This Dataset is a data structure provided by Hugging Face's Datasets library.

In [32]:
train_dataset = Dataset.from_dict(train_tokens) # converting tokenised data into hugging face object called dataset
val_dataset = Dataset.from_dict(val_tokens)
test_dataset = Dataset.from_dict(test_tokens)

# Data collator

In [33]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer)

A data collator prepares individual examples into a batch that can be fed to the model. The data collator takes a batch and dynamically pads them to the length of the longest example in that batch.

# Model Training

In [34]:
model = AutoModelForSequenceClassification.from_pretrained(
    "answerdotai/ModernBERT-base",
    num_labels=6,
    id2label={
        0: "ADHD",
        1: "Anxiety",
        2: "Bipolar",
        3: "Depression",
        4: "PTSD",
        5: "None"
    },
    label2id={
        "ADHD": 0,
        "Anxiety": 1,
        "Bipolar": 2,
        "Depression": 3,
        "PTSD": 4,
        "None": 5
    }
)

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ModernBERT loaded successfully; its pretrained layers are being reused, while a new classification head for your six classes has been initialized and will be learned during fine-tuning.

# Training configuration

In [35]:
training_args = TrainingArguments(
    output_dir="./modernbert_mhc",# older where Hugging Face will save training outputs.
    eval_strategy="epoch", # evaluation after each epoch
    save_strategy="epoch", # saving after each appoch
    learning_rate=2e-5, # rate at which weights are adjusted during training 
    per_device_train_batch_size=8,# GPU processes 8 training examples at a time.
    per_device_eval_batch_size=8, #GPU processes 8 validation examples at a time.
    num_train_epochs=3, # The model goes through the entire training dataset 3 times.
    weight_decay=0.01,# It encourages the model to avoid excessively large weights during optimization.
    logging_steps=100, # training report in every 100 steps
    load_best_model_at_end=True, ## best model is loaded at the end of training
    metric_for_best_model="f1", # f1 is chosen as the evaluation matric
    greater_is_better=True, # greater evaluation matrix is better
    report_to="none" # dont sent any training log to expternal agencies
)